In [2]:
from pathlib import Path
import pandas as pd
import numpy as np

# Define the path of your folder containing 60 zips
zip_folder = Path(r"C:\Users\ASUS\Desktop\tennis_data")

# Read proper tables
MatchEventInfo_df = pd.read_parquet(zip_folder / "event_all.parquet")

MatchHomeTeamInfo_df = pd.read_parquet(zip_folder / "home_team_all.parquet"
)

MatchAwayTeamInfo_df = pd.read_parquet(zip_folder / "away_team_all.parquet"
)

# Make a copy and clean data
MatchEventInfo_clean = MatchEventInfo_df[["match_id","winner_code"]].copy()

# Now, create home_country column
MatchEventInfo_clean = MatchEventInfo_clean.merge(
    MatchHomeTeamInfo_df[["match_id", "country"]],
    on="match_id",
    how="left"
)

MatchEventInfo_clean.rename(
    columns={"country" : "home_country"},
    inplace=True
)

# Now, create away_country column
MatchEventInfo_clean = MatchEventInfo_clean.merge(
    MatchAwayTeamInfo_df[["match_id", "country"]],
    on="match_id",
    how="left"
)

MatchEventInfo_clean.rename(
    columns={"country" : "away_country"},
    inplace=True
)

# Create winner country and drop winner code
MatchEventInfo_clean["winner_country"] = np.where(
    MatchEventInfo_clean["winner_code"] == 1,
    MatchEventInfo_clean["home_country"],
    MatchEventInfo_clean["away_country"]
)
MatchEventInfo_clean.drop("winner_code", axis=1, inplace=True)

# Drop duplicates
MatchEventInfo_clean.drop_duplicates(inplace=True)

In [3]:
# Now get number of wins and total number of matches to calculate success rate
winning_matches = (
    MatchEventInfo_clean["winner_country"]
    .value_counts()
    .rename("winning_matches")
)

total_matches = (
    pd.concat([
        MatchEventInfo_clean["home_country"],
        MatchEventInfo_clean["away_country"]
    ])
    .value_counts()
    .rename("total_matches")
)

country_info = pd.concat([winning_matches, total_matches],axis=1)

country_info["success_rate"] = country_info["winning_matches"] / country_info["total_matches"]

In [4]:
# Define score(a mix of success rate and number of matches)
# 1: Get the best country without omitting outliers from total_matches column
country_info["score"] = (
    country_info["success_rate"] *
    np.log10(country_info["total_matches"])
)
country_info.dropna(subset=["score"], inplace=True)

country_info.sort_values("score", ascending=False)

,winning_matches,total_matches,success_rate,score
France,1171.0,2139,0.547452,1.823131
Italy,1135.0,2067,0.549105,1.820470
USA,1005.0,1842,0.545603,1.781551
Russia,740.0,1306,0.566616,1.765542
Argentina,624.0,1089,0.573003,1.740225
...,...,...,...,...
Iran,2.0,9,0.222222,0.212054
Singapore,1.0,3,0.333333,0.159040
El Salvador,1.0,2,0.500000,0.150515
Barbados,1.0,2,0.500000,0.150515


In [7]:
# 2: Get the best country after omitting outliers from total_matches
Q1 = country_info["total_matches"].quantile(0.25)
Q3 = country_info["total_matches"].quantile(0.75)
IQR = Q3 - Q1

lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR

df = country_info[country_info["total_matches"].isna() |
    ((country_info["total_matches"] >= lower) &
     ( country_info["total_matches"]<= upper))
]

df.head()

,winning_matches,total_matches,success_rate,score
Brazil,328.0,617,0.531605,1.483328
Romania,315.0,561,0.561497,1.543535
Switzerland,289.0,521,0.554702,1.507037
China,280.0,522,0.536398,1.457754
Netherlands,264.0,488,0.540984,1.454391
